# 🛡️ VeriSlip: Dual-Stream Cross-Attention Network Training
### AI Forensic Fraud Detection for South Asian Bank Transfer Slips & Invoices

**Architecture Overview:**
- **Stream A:** RGB Spatial Visual Features
- **Stream B:** 3-Channel Forensic Tensor (Channel 0: ELA Error Map, Channel 1: High-Pass Noise Residual, Channel 2: DCT / Gradient Energy)
- **Cross-Modal Fusion:** Spatial Attention Gating
- **Dual Task Heads:** Binary Classification (Tampered vs Authentic) + Transpose Convolution Decoder (Pixel-Level Tamper Localization Mask)
- **Execution Target:** Kaggle GPU (Tesla T4 / P100 / Colab GPU)

In [ ]:
import os
import sys
import io
import glob
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from scipy.ndimage import median_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
from tqdm import tqdm

# Set seeds for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Running on device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 1. Dataset Extraction & Path Setup
Unpack the uploaded dataset archive if running on Kaggle.

In [ ]:
# Search for dataset folder or zip in Kaggle input or working directory
user_provided_path = '/kaggle/input/datasets/chirananavaratne/verislip-sl-banking-dataset/verislip_dataset'
DATASET_ROOT = None

# 1. Direct path check
candidates = [
    user_provided_path,
    '/kaggle/input/verislip-sl-banking-dataset/verislip_dataset',
    '/kaggle/input/verislip-sl-banking-dataset',
    'verislip_dataset'
]
for p in candidates:
    if os.path.exists(p) and os.path.exists(os.path.join(p, 'dataset_metadata.csv')):
        DATASET_ROOT = p
        break

# 2. Recursive search across /kaggle/input and current workspace
if not DATASET_ROOT:
    csv_candidates = glob.glob('/kaggle/input/**/dataset_metadata.csv', recursive=True) + glob.glob('**/dataset_metadata.csv', recursive=True)
    if csv_candidates:
        DATASET_ROOT = os.path.dirname(csv_candidates[0])

# 3. Check for zip archive if uploaded as zip
if not DATASET_ROOT:
    possible_zips = glob.glob('/kaggle/input/**/*.zip', recursive=True) + glob.glob('*.zip')
    if possible_zips:
        zip_path = possible_zips[0]
        print(f'📦 Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('.')
        print('✓ Extraction complete.')
        csv_candidates = glob.glob('**/dataset_metadata.csv', recursive=True)
        if csv_candidates:
            DATASET_ROOT = os.path.dirname(csv_candidates[0])

if DATASET_ROOT:
    csv_path = os.path.join(DATASET_ROOT, 'dataset_metadata.csv')
    df_meta = pd.read_csv(csv_path)
    print(f'✓ Found dataset at: {DATASET_ROOT}')
    print(f'✓ Loaded metadata with {len(df_meta)} samples.')
    display(df_meta.head(4))
else:
    print('⚠️ Please ensure your dataset is attached in Kaggle "Add Data" or check input directory.')


## 2. On-the-Fly 3-Channel Forensic Tensor Extractor
Calculates ELA, Noise Residual, and Spatial Gradient Discontinuity for every sample.

In [ ]:
def extract_forensic_tensor(pil_img, target_wh=(256, 256)):
    """
    Extract 3-channel forensic map tensor:
    Channel 0: Error Level Analysis (ELA) recompression difference
    Channel 1: Spatial High-Pass Noise Residual (Median Filter Residual)
    Channel 2: Sobel Edge / Gradient Discontinuity
    """
    tw, th = target_wh
    resized = pil_img.resize((tw, th), Image.BILINEAR)
    rgb_arr = np.array(resized, dtype=np.float32)
    gray = cv2.cvtColor(rgb_arr.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)

    # Channel 0: ELA Error Map
    buf = io.BytesIO()
    resized.save(buf, format='JPEG', quality=85)
    buf.seek(0)
    recompressed = Image.open(buf)
    recomp_arr = np.array(recompressed, dtype=np.float32)
    ela_diff = np.mean(np.abs(rgb_arr - recomp_arr), axis=2)
    ela_ch = np.clip(ela_diff / 25.0, 0.0, 1.0)

    # Channel 1: High-Pass Noise Residual
    denoised = median_filter(gray, size=3)
    noise = np.abs(gray - denoised)
    noise_ch = np.clip(noise / 18.0, 0.0, 1.0)

    # Channel 2: Spatial Gradient
    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(sobelx**2 + sobely**2)
    grad_ch = np.clip(grad / 60.0, 0.0, 1.0)

    # Stack into (3, H, W)
    forensic_tensor = np.stack([ela_ch, noise_ch, grad_ch], axis=0).astype(np.float32)
    return forensic_tensor

class VeriSlipDataset(Dataset):
    def __init__(self, root_dir, split='train', img_size=(256, 256)):
        self.root_dir = root_dir
        self.split = split
        self.img_size = img_size
        
        csv_path = os.path.join(root_dir, 'dataset_metadata.csv')
        df = pd.read_csv(csv_path)
        self.samples = df[df['split'] == split].reset_index(drop=True)
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        row = self.samples.iloc[idx]
        img_rel_path = row['filename']
        mask_rel_path = row['mask_filename']
        
        img_full_path = os.path.join(self.root_dir, img_rel_path)
        mask_full_path = os.path.join(self.root_dir, mask_rel_path)
        
        pil_img = Image.open(img_full_path).convert('RGB')
        forensic_tensor = extract_forensic_tensor(pil_img, self.img_size)
        
        # Resize RGB image and normalize to [0, 1]
        rgb_resized = pil_img.resize(self.img_size, Image.BILINEAR)
        rgb_tensor = np.array(rgb_resized, dtype=np.float32).transpose(2, 0, 1) / 255.0
        
        # Load and resize mask to (1, H, W)
        mask_pil = Image.open(mask_full_path).convert('L')
        mask_resized = mask_pil.resize(self.img_size, Image.NEAREST)
        mask_tensor = (np.array(mask_resized, dtype=np.float32) > 128).astype(np.float32)[np.newaxis, ...]
        
        label = float(row['is_tampered'])
        
        return (
            torch.tensor(rgb_tensor, dtype=torch.float32),
            torch.tensor(forensic_tensor, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
            torch.tensor(mask_tensor, dtype=torch.float32)
        )

# Create Datasets and DataLoaders
train_ds = VeriSlipDataset(DATASET_ROOT, split='train')
val_ds = VeriSlipDataset(DATASET_ROOT, split='val')

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'✓ Train Samples: {len(train_ds)} | Val Samples: {len(val_ds)}')

## 3. Dual-Stream Forensic Neural Network Definition

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.MaxPool2d(2, 2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(self.conv(x))

class DualStreamForensicNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # Stream A: RGB Visual Features (3 -> 32 -> 64 -> 128)
        self.rgb_stream = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128)
        )

        # Stream B: Forensic Tensor Features (3 -> 32 -> 64 -> 128)
        self.forensic_stream = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128)
        )

        # Spatial Attention Gate
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(256, 64, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.Sigmoid()
        )

        # Classification Head
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )

        # Localization Segmentation Decoder Head (Upsample 8x back to 256x256)
        self.loc_decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1),  # 2x
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # 4x
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),   # 8x
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=3, padding=1)
        )

    def forward(self, rgb_tensor, forensic_tensor):
        feat_rgb = self.rgb_stream(rgb_tensor)
        feat_forensic = self.forensic_stream(forensic_tensor)
        
        fused = torch.cat([feat_rgb, feat_forensic], dim=1)  # (B, 256, H/8, W/8)
        gate = self.spatial_gate(fused)
        gated = fused * gate
        
        # Classification Logits
        pooled = self.avg_pool(gated).flatten(1)
        cls_logits = self.classifier(pooled).squeeze(1)
        
        # Localization Logits
        loc_logits = self.loc_decoder(gated)
        
        return cls_logits, loc_logits

model = DualStreamForensicNetwork().to(device)
print('✓ DualStreamForensicNetwork initialized successfully!')

## 4. Multi-Task Loss Functions: BCE + Dice Loss

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.contiguous().view(-1)
        targets = targets.contiguous().view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

criterion_cls = nn.BCEWithLogitsLoss()
criterion_loc = DiceLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-5)

## 5. Training & Validation Loop
Trains model for 15 epochs with real-time accuracy and Dice IoU tracking.

In [ ]:
EPOCHS = 15
best_val_f1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': []}

print(f'🏋️ Starting training for {EPOCHS} epochs on {device}...')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    
    for rgb, forensic, labels, masks in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]'):
        rgb = rgb.to(device)
        forensic = forensic.to(device)
        labels = labels.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        cls_logits, loc_logits = model(rgb, forensic)
        
        loss_cls = criterion_cls(cls_logits, labels)
        loss_loc = criterion_loc(loc_logits, masks)
        total_loss = loss_cls + 1.8 * loss_loc
        
        total_loss.backward()
        optimizer.step()
        train_losses.append(total_loss.item())
        
    scheduler.step()
    avg_train_loss = np.mean(train_losses)
    
    # Validation Loop
    model.eval()
    val_losses = []
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for rgb, forensic, labels, masks in val_loader:
            rgb = rgb.to(device)
            forensic = forensic.to(device)
            labels = labels.to(device)
            masks = masks.to(device)
            
            cls_logits, loc_logits = model(rgb, forensic)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_loc = criterion_loc(loc_logits, masks)
            val_losses.append((loss_cls + 1.8 * loss_loc).item())
            
            probs = torch.sigmoid(cls_logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    avg_val_loss = np.mean(val_losses)
    val_acc = accuracy_score(all_labels, all_preds)
    _, _, val_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', zero_division=0)
    try:
        val_auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        val_auc = 0.5
        
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['val_auc'].append(val_auc)
    
    print(f'Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val F1: {val_f1:.4f} | AUC: {val_auc:.4f}')
    
    # Save best checkpoint
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'verislip_dualstream_best.pt')
        print(f'  ⭐ New best checkpoint saved with F1: {val_f1:.4f}')

print('\n✅ Training successfully completed! Best model saved to verislip_dualstream_best.pt')

## 6. Training Curves & Visual Anomaly Localization Test

In [ ]:
# Plot Loss & Accuracy Curves
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history['val_acc'], label='Val Accuracy', color='green')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history['val_f1'], label='Val F1 Score', color='purple')
plt.plot(history['val_auc'], label='ROC-AUC', color='orange')
plt.title('F1 Score & ROC-AUC')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()

# Visual Inspection of Sample Test Slip
model.load_state_dict(torch.load('verislip_dualstream_best.pt'))
model.eval()

rgb, forensic, label, mask = val_ds[0]
with torch.no_grad():
    c_log, l_log = model(rgb.unsqueeze(0).to(device), forensic.unsqueeze(0).to(device))
    prob = torch.sigmoid(c_log).item()
    pred_mask = torch.sigmoid(l_log).squeeze().cpu().numpy()

plt.figure(figsize=(12, 4))
plt.subplot(1, 4, 1)
plt.imshow(rgb.numpy().transpose(1, 2, 0))
plt.title(f'Original (Label: {int(label.item())})')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(forensic.numpy()[0], cmap='inferno')
plt.title('ELA Heatmap Stream')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(mask.squeeze().numpy(), cmap='gray')
plt.title('Ground-Truth Mask')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(pred_mask, cmap='hot')
plt.title(f'Pred Tamper (Prob: {prob*100:.1f}%)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Package and Download Weights ZIP
Run this cell to package the trained model checkpoint and click the generated link to download directly in your browser.

In [ ]:
import os
import zipfile
import json
from IPython.display import FileLink, display, HTML

zip_filename = 'verislip_trained_weights.zip'

# Save metrics ledger
if 'history' in globals():
    with open('training_metrics.json', 'w') as f:
        json.dump(history, f, indent=2)

# Package weights and metrics into zip
with zipfile.ZipFile(zip_filename, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    if os.path.exists('verislip_dualstream_best.pt'):
        z.write('verislip_dualstream_best.pt', arcname='verislip_dualstream_best.pt')
        print('✓ Added verislip_dualstream_best.pt to zip archive')
    if os.path.exists('training_metrics.json'):
        z.write('training_metrics.json', arcname='training_metrics.json')
        print('✓ Added training_metrics.json to zip archive')

print(f'\n📦 {zip_filename} created successfully!')
print('Click the link below to download your weights:')
display(FileLink(zip_filename))
display(HTML(f'''<a href="{zip_filename}" download style="display:inline-block; margin-top:10px; padding:10px 20px; background:#10b981; color:white; border-radius:6px; text-decoration:none; font-weight:bold;">⬇️ Download {zip_filename}</a>'''))


## 8. How to Activate Model in Your Local VeriSlip App
1. Unzip `verislip_trained_weights.zip` (or extract `verislip_dualstream_best.pt`).
2. Place `verislip_dualstream_best.pt` in your local repo at: `weights/verislip_dualstream_best.pt`.
3. Start or reload VeriSlip — Layer 4 will automatically load your trained model weights for real-time verification!